# EBA 3650 — Carbon Tax Revenue Recycling
**Group 6 | Puskar Acharya, Peien Chen, Qianqian Cui**

**Research Question:** Which revenue recycling mechanism — universal lump-sum redistribution or targeted compensation — more effectively mitigates the regressive effects of carbon taxation, and what is the associated efficiency cost in terms of aggregate labour supply?

---

### Notebook Structure
1. Imports and Parameters  
2. Wage Distribution Simulation and Gini Calibration  
3. Household Optimisation Function  
4. Baseline: No Redistribution  
5. Scheme A: Universal Lump-Sum Rebate (Fixed-Point Iteration)  
6. Scheme B: Targeted Compensation (Fixed-Point Iteration)  
7. Newton's Method Robustness Check  
8. Outcome Variables and Decile Analysis  
9. Sensitivity Analysis (τ and λ grids)  
10. Figures (all six output charts)

---
## Section 1 — Imports and Parameters

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import minimize_scalar, newton
import warnings
warnings.filterwarnings('ignore')

# ── Baseline parameters ─────────────────────────────────────────────────────
params = {
    "tau"  : 0.10,    # carbon tax rate (ad valorem on energy)
    "beta" : 0.15,    # energy share of consumption
    "alpha": 0.50,    # leisure preference in utility function
    "lam"  : 1.50,    # targeting multiplier for bottom 40% (Scheme B)
    "N"    : 1000,    # number of simulated households
    "tol"  : 1e-6,    # convergence tolerance for fixed-point loop
    "seed" : 42,      # random seed for reproducibility
}

# ── Log-normal wage distribution parameters (calibrated to Gini ≈ 0.35) ────
MU_W    = 0.0    # mean of log wages
SIGMA_W = 0.75   # std dev of log wages — adjust until Gini ≈ 0.35

print("Parameters loaded.")
print(pd.Series(params).to_string())

ModuleNotFoundError: No module named 'scipy'

---
## Section 2 — Wage Distribution Simulation and Gini Calibration

We draw N = 1,000 wages from a log-normal distribution and calibrate σ so that the
resulting Gini coefficient matches the European target of approximately 0.35.

In [ ]:
def compute_gini(x):
    """
    Compute Gini coefficient via numerical integration of the Lorenz curve.
    Uses np.trapezoid — consistent with Week 6 (numerical integration).

    Parameters
    ----------
    x : array-like, unsorted income/wage values

    Returns
    -------
    float : Gini coefficient in [0, 1]
    """
    x_sorted = np.sort(x)
    n = len(x_sorted)
    cum_share = np.cumsum(x_sorted) / x_sorted.sum()
    lorenz = np.concatenate([[0.0], cum_share])   # Lorenz curve starts at (0,0)
    pop_share = np.linspace(0, 1, n + 1)
    area_under_lorenz = np.trapezoid(lorenz, pop_share)
    return 1.0 - 2.0 * area_under_lorenz


def calibrate_lognormal(target_gini=0.35, N=1000, seed=42,
                         mu=0.0, sigma_init=0.75,
                         tol=0.005, max_iter=50):
    """
    Adjust sigma of the log-normal until the simulated Gini
    is within tol of the target.

    Returns sorted wages and the calibrated sigma.
    """
    sigma = sigma_init
    rng = np.random.default_rng(seed)
    for i in range(max_iter):
        w = np.sort(rng.lognormal(mean=mu, sigma=sigma, size=N))
        g = compute_gini(w)
        if abs(g - target_gini) < tol:
            print(f"Calibrated in {i+1} steps:  sigma = {sigma:.4f},  Gini = {g:.4f}")
            return w, sigma
        # Gini rises with sigma for log-normal — simple proportional update
        sigma *= target_gini / g
    print(f"Warning: did not converge. Final Gini = {g:.4f}")
    return w, sigma


# ── Run calibration ──────────────────────────────────────────────────────────
wages, sigma_calibrated = calibrate_lognormal(
    target_gini=0.35,
    N=params["N"],
    seed=params["seed"],
    mu=MU_W
)

# ── Assign decile labels (0 = bottom, 9 = top) ──────────────────────────────
decile_labels = pd.qcut(wages, q=10, labels=False)   # 0-indexed

print(f"\nWage stats:")
print(f"  Min  : {wages.min():.3f}")
print(f"  Mean : {wages.mean():.3f}")
print(f"  Max  : {wages.max():.3f}")
print(f"  Gini : {compute_gini(wages):.4f}  (target 0.35)")

---
## Section 3 — Household Optimisation Function

Each household i solves:

    max  U(c, l) = log(c) + α·log(1 - l)
    s.t. c·(1 + τ·β) = w_i·l + T_i

The energy demand rule e = β·c is substituted into the budget constraint,
reducing the problem to a one-dimensional optimisation over l ∈ (0, 1).

We solve numerically with `scipy.optimize.minimize_scalar` and verify with
the analytical first-order condition.

In [ ]:
# ── Analytical solution from the FOC ────────────────────────────────────────
#
# From the Lagrangian, the FOC for labour supply is:
#   (1/c) · w_i = α / (1 - l)
#
# Substituting the budget constraint c = (w_i·l + T_i) / (1 + τ·β):
#
#   w_i · (1 + τ·β) / (w_i·l + T_i) = α / (1 - l)
#
# Solving for l*:
#   w_i·(1-l)·(1+τ·β) = α·(w_i·l + T_i)
#   w_i·(1+τ·β) - w_i·(1+τ·β)·l = α·w_i·l + α·T_i
#   w_i·(1+τ·β) - α·T_i = l·[w_i·(1+τ·β) + α·w_i]
#   l* = [w_i·(1+τ·β) - α·T_i] / [w_i·(1+τ·β+α)]

def labour_analytical(w, T, tau, beta, alpha):
    """
    Closed-form optimal labour supply from the FOC of log-log utility.

    Parameters
    ----------
    w     : float, household wage
    T     : float, government transfer
    tau   : float, carbon tax rate
    beta  : float, energy expenditure share
    alpha : float, leisure preference

    Returns
    -------
    l_star : float, optimal labour in (0, 1)
    """
    p = 1.0 + tau * beta          # effective price index
    numerator   = w * p - alpha * T
    denominator = w * (p + alpha)
    l_star = numerator / denominator
    return np.clip(l_star, 1e-6, 1.0 - 1e-6)


def solve_household(w, T, tau, beta, alpha, method="analytical"):
    """
    Solve one household's utility maximisation problem.

    Parameters
    ----------
    w, T, tau, beta, alpha : floats (see above)
    method : "analytical" (default) or "numerical" (scipy verification)

    Returns
    -------
    l_star, c_star, e_star : optimal labour, consumption, energy
    """
    p = 1.0 + tau * beta

    if method == "analytical":
        l_star = labour_analytical(w, T, tau, beta, alpha)

    else:  # numerical — used for verification
        def neg_utility(l):
            if l <= 0 or l >= 1:
                return 1e10
            c = (w * l + T) / p
            if c <= 0:
                return 1e10
            return -(np.log(c) + alpha * np.log(1.0 - l))

        result = minimize_scalar(neg_utility, bounds=(1e-4, 1 - 1e-4),
                                 method="bounded")
        l_star = result.x

    c_star = (w * l_star + T) / p
    e_star = beta * c_star
    return l_star, c_star, e_star


def solve_all_households(wages, transfers, params, method="analytical"):
    """
    Vectorised wrapper: solve all N households.

    Parameters
    ----------
    wages     : array of shape (N,), sorted ascending
    transfers : array of shape (N,), one transfer per household
    params    : dict of model parameters
    method    : "analytical" or "numerical"

    Returns
    -------
    l_star, c_star, e_star : arrays of shape (N,)
    """
    tau, beta, alpha = params["tau"], params["beta"], params["alpha"]
    N = len(wages)
    l_star = np.zeros(N)
    c_star = np.zeros(N)
    e_star = np.zeros(N)
    for i in range(N):
        l_star[i], c_star[i], e_star[i] = solve_household(
            wages[i], transfers[i], tau, beta, alpha, method=method
        )
    return l_star, c_star, e_star


# ── Quick verification: analytical vs numerical for one household ────────────
w_test, T_test = 1.5, 0.05
l_a, c_a, e_a = solve_household(w_test, T_test,
                                params["tau"], params["beta"], params["alpha"],
                                method="analytical")
l_n, c_n, e_n = solve_household(w_test, T_test,
                                params["tau"], params["beta"], params["alpha"],
                                method="numerical")

print("Verification (w=1.5, T=0.05):")
print(f"  Analytical:  l* = {l_a:.6f},  c* = {c_a:.6f},  e* = {e_a:.6f}")
print(f"  Numerical:   l* = {l_n:.6f},  c* = {c_n:.6f},  e* = {e_n:.6f}")
print(f"  Max abs diff in l*: {abs(l_a - l_n):.2e}")

---
## Section 4 — Baseline: No Redistribution

All households receive T = 0. This is the reference point for welfare comparisons.
It also demonstrates the raw regressivity of the carbon tax before any recycling.

In [ ]:
# ── Solve baseline (T = 0 for everyone) ─────────────────────────────────────
T_zero = np.zeros(params["N"])
l_base, c_base, e_base = solve_all_households(wages, T_zero, params)

# Baseline utility
U_base = np.log(c_base) + params["alpha"] * np.log(1.0 - l_base)

# Baseline tax paid and pre-tax labour income
tax_base    = params["tau"] * e_base
income_base = wages * l_base

# Tax burden share by decile: tax paid / pre-tax income
burden_base = np.zeros(10)
for d in range(10):
    mask = decile_labels == d
    burden_base[d] = tax_base[mask].sum() / income_base[mask].sum()

# Aggregate labour supply
L_base = l_base.sum()

print("Baseline (no redistribution):")
print(f"  Aggregate labour supply : {L_base:.2f}")
print(f"  Total tax revenue       : {tax_base.sum():.4f}")
print(f"  Mean utility            : {U_base.mean():.4f}")
print()
print("Tax burden share by decile (demonstrates regressivity):")
for d in range(10):
    print(f"  Decile {d+1:2d}: {burden_base[d]:.4f}")

---
## Section 5 — Scheme A: Universal Lump-Sum Rebate (Fixed-Point Iteration)

All N households receive the same transfer T = R / N, where R is total tax revenue.

The fixed-point loop (Weeks 2-3):
1. Start with T⁰ = 0
2. Solve all households under current T
3. Compute new revenue R, update T = R / N
4. Repeat until |T_new - T_old| < tol

In [ ]:
def solve_scheme_A(wages, params, verbose=True):
    """
    Scheme A: Universal lump-sum rebate.

    Every household receives T = total_revenue / N.
    Solved via fixed-point iteration on the government budget constraint.

    Parameters
    ----------
    wages  : array of shape (N,), sorted ascending
    params : dict of model parameters

    Returns
    -------
    l_star, c_star, e_star : arrays of shape (N,)
    T_eq   : float, equilibrium transfer
    history: list of (iteration, T_old, T_new) for convergence tracking
    """
    tau  = params["tau"]
    N    = params["N"]
    tol  = params["tol"]

    T       = 0.0     # initial guess
    history = []

    for iteration in range(1000):
        transfers = np.full(N, T)
        l_star, c_star, e_star = solve_all_households(wages, transfers, params)

        revenue = tau * e_star.sum()
        T_new   = revenue / N

        history.append((iteration + 1, T, T_new))

        if abs(T_new - T) < tol:
            if verbose:
                print(f"Scheme A converged in {iteration+1} iterations.")
                print(f"  Equilibrium transfer T = {T_new:.6f}")
                print(f"  Total revenue          = {revenue:.4f}")
            return l_star, c_star, e_star, T_new, history
        T = T_new

    raise RuntimeError("Scheme A: fixed-point iteration did not converge.")


# ── Run Scheme A ─────────────────────────────────────────────────────────────
l_A, c_A, e_A, T_A, history_A = solve_scheme_A(wages, params)

U_A = np.log(c_A) + params["alpha"] * np.log(1.0 - l_A)
L_A = l_A.sum()

print(f"\nScheme A — Aggregate labour supply: {L_A:.4f}")
print(f"Scheme A — Mean utility           : {U_A.mean():.6f}")

---
## Section 6 — Scheme B: Targeted Compensation (Fixed-Point Iteration)

Bottom 40% of households (by wage rank) receive T_low = λ·T̄.
Top 60% receive T_high determined by the budget constraint:

    0.4·N·T_low + 0.6·N·T_high = Total Revenue

The fixed-point loop iterates on T̄ (mean rebate) until convergence.

In [ ]:
def solve_scheme_B(wages, params, lam=None, verbose=True):
    """
    Scheme B: Targeted compensation.

    Bottom 40% get T_low = lam * T_mean.
    Top 60% get T_high from the budget constraint.

    Parameters
    ----------
    wages  : array of shape (N,), sorted ascending
    params : dict of model parameters
    lam    : float, targeting multiplier (overrides params["lam"] if given)

    Returns
    -------
    l_star, c_star, e_star : arrays of shape (N,)
    T_low, T_high : equilibrium transfers for the two groups
    history       : convergence history
    """
    tau  = params["tau"]
    N    = params["N"]
    tol  = params["tol"]
    if lam is None:
        lam = params["lam"]

    cutoff       = int(0.40 * N)          # first 40% (wages sorted ascending)
    n_bottom     = cutoff
    n_top        = N - cutoff
    bottom_mask  = np.arange(N) < cutoff  # True for bottom 40%

    T_mean   = 0.0    # initial guess for mean rebate
    history  = []

    for iteration in range(1000):
        T_low  = lam * T_mean

        # From budget: 0.4*N*T_low + 0.6*N*T_high = revenue
        # But revenue depends on T_high, so we use T_mean to set T_high:
        # total_transfer = N * T_mean  =>  T_high = (N*T_mean - n_bottom*T_low) / n_top
        total_transfer = N * T_mean
        T_high = (total_transfer - n_bottom * T_low) / n_top
        T_high = max(T_high, 0.0)  # transfers cannot be negative

        transfers = np.where(bottom_mask, T_low, T_high)
        l_star, c_star, e_star = solve_all_households(wages, transfers, params)

        revenue    = tau * e_star.sum()
        T_mean_new = revenue / N

        history.append((iteration + 1, T_mean, T_mean_new))

        if abs(T_mean_new - T_mean) < tol:
            if verbose:
                print(f"Scheme B (λ={lam}) converged in {iteration+1} iterations.")
                print(f"  T_mean = {T_mean_new:.6f}")
                print(f"  T_low  = {lam*T_mean_new:.6f}  (bottom 40%)")
                print(f"  T_high = {T_high:.6f}  (top 60%)")
                print(f"  Total revenue = {revenue:.4f}")
            return l_star, c_star, e_star, lam * T_mean_new, T_high, history

        T_mean = T_mean_new

    raise RuntimeError("Scheme B: fixed-point iteration did not converge.")


# ── Run Scheme B ─────────────────────────────────────────────────────────────
l_B, c_B, e_B, T_B_low, T_B_high, history_B = solve_scheme_B(wages, params)

U_B = np.log(c_B) + params["alpha"] * np.log(1.0 - l_B)
L_B = l_B.sum()

print(f"\nScheme B — Aggregate labour supply: {L_B:.4f}")
print(f"Scheme B — Mean utility           : {U_B.mean():.6f}")

---
## Section 7 — Newton's Method Robustness Check

Reformulate the Scheme A equilibrium as a root-finding problem:

    f(T) = T - R(T)/N = 0

Solve with `scipy.optimize.newton` and verify against the fixed-point result.

In [ ]:
def budget_residual(T_val, wages, params):
    """
    Root-finding formulation of the Scheme A equilibrium.
    f(T) = T - R(T)/N = 0  at equilibrium.

    Parameters
    ----------
    T_val  : float, candidate transfer
    wages  : array of shape (N,)
    params : dict

    Returns
    -------
    float : residual f(T)
    """
    N         = params["N"]
    tau       = params["tau"]
    transfers = np.full(N, T_val)
    _, _, e_star = solve_all_households(wages, transfers, params)
    revenue   = tau * e_star.sum()
    return T_val - revenue / N


# ── Solve with Newton's method ───────────────────────────────────────────────
T_newton = newton(
    budget_residual,
    x0  = 0.01,
    args= (wages, params),
    tol = 1e-8,
    maxiter = 100
)

print("Newton's method robustness check (Scheme A):")
print(f"  Fixed-point result : T = {T_A:.8f}")
print(f"  Newton result      : T = {T_newton:.8f}")
print(f"  Difference         :     {abs(T_A - T_newton):.2e}")

if abs(T_A - T_newton) < 1e-5:
    print("  PASSED: both methods agree.")
else:
    print("  WARNING: methods disagree — check implementation.")

---
## Section 8 — Outcome Variables and Decile Analysis

For each scenario (Baseline, Scheme A, Scheme B) we compute four outcome variables
by income decile:

1. Net welfare change  ΔU_d
2. Tax burden share    B_d
3. Share of winners   W_d
4. Aggregate labour supply L

In [ ]:
def compute_decile_outcomes(l_star, c_star, e_star, U_base, wages,
                             decile_labels, params, label="Scenario"):
    """
    Compute the four outcome variables for each of 10 deciles.

    Parameters
    ----------
    l_star, c_star, e_star : arrays of shape (N,), household choices under scenario
    U_base       : array of shape (N,), baseline (no-tax) utility for each household
    wages        : array of shape (N,), household wages
    decile_labels: array of shape (N,), decile index 0-9
    params       : dict
    label        : str, name of the scenario (for display)

    Returns
    -------
    pd.DataFrame with columns: decile, delta_U, burden, winners, labour
    """
    alpha = params["alpha"]
    tau   = params["tau"]

    # Current utility under scenario
    U_new   = np.log(c_star) + alpha * np.log(1.0 - l_star)
    delta_U = U_new - U_base

    income = wages * l_star   # labour income under scenario

    rows = []
    for d in range(10):
        mask = decile_labels == d
        rows.append({
            "decile" : d + 1,
            "delta_U": delta_U[mask].mean(),
            "burden" : (tau * e_star[mask]).sum() / income[mask].sum(),
            "winners": (delta_U[mask] > 0).mean(),
            "labour" : l_star[mask].sum(),
        })

    df = pd.DataFrame(rows)
    df.attrs["label"]  = label
    df.attrs["L_total"]= l_star.sum()
    return df


# ── Compute outcomes for all three scenarios ─────────────────────────────────
df_base = compute_decile_outcomes(l_base, c_base, e_base, U_base,
                                  wages, decile_labels, params, "Baseline")

df_A    = compute_decile_outcomes(l_A,    c_A,    e_A,    U_base,
                                  wages, decile_labels, params, "Scheme A")

df_B    = compute_decile_outcomes(l_B,    c_B,    e_B,    U_base,
                                  wages, decile_labels, params, "Scheme B")


# ── Summary table ────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    "Decile"            : df_A["decile"],
    "ΔU (Scheme A)"     : df_A["delta_U"].round(4),
    "ΔU (Scheme B)"     : df_B["delta_U"].round(4),
    "Burden (Baseline)" : df_base["burden"].round(4),
    "Burden (A)"        : df_A["burden"].round(4),
    "Burden (B)"        : df_B["burden"].round(4),
    "Winners A (%)"     : (df_A["winners"] * 100).round(1),
    "Winners B (%)"     : (df_B["winners"] * 100).round(1),
})

print("=" * 80)
print("DECILE OUTCOME SUMMARY")
print("=" * 80)
print(summary.to_string(index=False))
print()
print(f"Aggregate Labour Supply:")
print(f"  Baseline : {L_base:.4f}")
print(f"  Scheme A : {L_A:.4f}  (change: {L_A - L_base:+.4f})")
print(f"  Scheme B : {L_B:.4f}  (change: {L_B - L_base:+.4f})")

---
## Section 9 — Sensitivity Analysis (τ and λ grids)

We vary the carbon tax rate τ ∈ {0.05, 0.10, 0.15, 0.20} and the targeting
multiplier λ ∈ {1.25, 1.50, 1.75, 2.00} independently.

For each combination we record:
- Mean welfare gain of the bottom 40% of households
- Aggregate labour supply loss relative to baseline

In [ ]:
tau_grid = [0.05, 0.10, 0.15, 0.20]
lam_grid = [1.25, 1.50, 1.75, 2.00]

frontier_rows = []

for lam in lam_grid:
    for tau in tau_grid:
        p = {**params, "tau": tau, "lam": lam}

        # Recompute baseline labour for this tau
        l_b0, c_b0, e_b0 = solve_all_households(wages, T_zero, p)
        U_b0 = np.log(c_b0) + p["alpha"] * np.log(1.0 - l_b0)

        # Scheme B under (tau, lam)
        try:
            l_s, c_s, e_s, _, _, _ = solve_scheme_B(
                wages, p, lam=lam, verbose=False
            )
        except RuntimeError:
            print(f"Did not converge: tau={tau}, lam={lam}")
            continue

        U_s = np.log(c_s) + p["alpha"] * np.log(1.0 - l_s)

        cutoff      = int(0.40 * params["N"])
        bottom_mask = np.arange(params["N"]) < cutoff

        welfare_bottom = (U_s[bottom_mask] - U_b0[bottom_mask]).mean()
        labour_loss    = l_s.sum() - l_b0.sum()

        frontier_rows.append({
            "tau"             : tau,
            "lam"             : lam,
            "welfare_bottom40": welfare_bottom,
            "labour_loss"     : labour_loss,
        })

frontier_df = pd.DataFrame(frontier_rows)

print("Sensitivity grid (Scheme B, bottom 40% welfare gain and labour loss):")
print(frontier_df.to_string(index=False))

---
## Section 10 — Figures

All six output figures as specified in the project proposal (Section 12).

- Figure 1: Simulated Wage Distribution  
- Figure 2: Carbon Tax Burden by Decile (No Redistribution)  
- Figure 3: Net Welfare Change by Decile (Scheme A vs Scheme B)  
- Figure 4: Share of Winners by Decile  
- Figure 5: Aggregate Labour Supply Comparison  
- Figure 6: Equity-Efficiency Frontier (λ sensitivity)

In [ ]:
import os
os.makedirs("figures", exist_ok=True)

COLORS = {
    "baseline": "#555555",
    "scheme_a" : "#2196F3",
    "scheme_b" : "#E53935",
    "grid"     : "#e0e0e0",
}

DECILES = np.arange(1, 11)

plt.rcParams.update({
    "figure.dpi"    : 130,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "font.size"     : 11,
})

In [ ]:
# ── Figure 1: Simulated Wage Distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Histogram
ax = axes[0]
ax.hist(wages, bins=40, color=COLORS["scheme_a"], edgecolor="white", linewidth=0.5, alpha=0.85)
ax.set_xlabel("Wage (w)")
ax.set_ylabel("Count")
ax.set_title("Figure 1a: Simulated Wage Distribution", fontweight="bold")
ax.text(0.97, 0.95,
        f"N = {params['N']}\nGini = {compute_gini(wages):.3f}\nσ = {sigma_calibrated:.3f}",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#cccccc"))

# Lorenz curve
ax2 = axes[1]
w_sorted  = np.sort(wages)
cum_pop   = np.linspace(0, 1, params["N"] + 1)
cum_wage  = np.concatenate([[0], np.cumsum(w_sorted) / w_sorted.sum()])
ax2.plot(cum_pop, cum_pop,   color="#999999", linestyle="--", label="Line of equality")
ax2.plot(cum_pop, cum_wage,  color=COLORS["scheme_a"], linewidth=2, label="Lorenz curve")
ax2.fill_between(cum_pop, cum_pop, cum_wage, alpha=0.15, color=COLORS["scheme_a"])
ax2.set_xlabel("Cumulative population share")
ax2.set_ylabel("Cumulative wage share")
ax2.set_title("Figure 1b: Lorenz Curve", fontweight="bold")
ax2.legend()

plt.tight_layout()
plt.savefig("figures/fig1_wage_distribution.png", bbox_inches="tight")
plt.show()
print("Figure 1 saved.")

In [ ]:
# ── Figure 2: Tax Burden by Decile (No Redistribution) ──────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(DECILES, burden_base * 100, color=COLORS["baseline"],
              edgecolor="white", linewidth=0.6)

for bar, val in zip(bars, burden_base * 100):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val:.2f}%", ha="center", va="bottom", fontsize=9)

ax.set_xlabel("Income Decile (1 = poorest)")
ax.set_ylabel("Tax Burden Share (% of income)")
ax.set_title("Figure 2: Carbon Tax Burden by Decile — No Redistribution",
             fontweight="bold")
ax.set_xticks(DECILES)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.set_ylim(0, burden_base.max() * 100 * 1.2)
ax.annotate("Tax is regressive: burden falls more heavily on lower-income deciles.",
            xy=(0.02, 0.97), xycoords="axes fraction",
            va="top", fontsize=9, color="#444444",
            bbox=dict(boxstyle="round,pad=0.3", fc="#f5f5f5", ec="none"))

plt.tight_layout()
plt.savefig("figures/fig2_tax_burden_baseline.png", bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

In [ ]:
# ── Figure 3: Net Welfare Change by Decile — Scheme A vs Scheme B ────────────
x     = np.arange(10)
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5.5))

bars_A = ax.bar(x - width / 2, df_A["delta_U"], width,
                label="Scheme A (lump-sum)",
                color=COLORS["scheme_a"], edgecolor="white", linewidth=0.5)
bars_B = ax.bar(x + width / 2, df_B["delta_U"], width,
                label="Scheme B (targeted)",
                color=COLORS["scheme_b"], edgecolor="white", linewidth=0.5)

ax.axhline(0, color="black", linewidth=0.8, linestyle="-")
ax.set_xlabel("Income Decile (1 = poorest)")
ax.set_ylabel("Mean Welfare Change (ΔU)")
ax.set_title("Figure 3: Net Welfare Change by Decile — Scheme A vs Scheme B",
             fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([f"D{d}" for d in DECILES])
ax.legend(frameon=True)
ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.7)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("figures/fig3_welfare_change.png", bbox_inches="tight")
plt.show()
print("Figure 3 saved.")

In [ ]:
# ── Figure 4: Share of Winners by Decile ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(x - width / 2, df_A["winners"] * 100, width,
       label="Scheme A", color=COLORS["scheme_a"], alpha=0.9, edgecolor="white")
ax.bar(x + width / 2, df_B["winners"] * 100, width,
       label="Scheme B", color=COLORS["scheme_b"], alpha=0.9, edgecolor="white")

ax.axhline(50, color="#666666", linewidth=0.8, linestyle="--", label="50% threshold")
ax.set_xlabel("Income Decile (1 = poorest)")
ax.set_ylabel("Share of households with ΔU > 0 (%)")
ax.set_title("Figure 4: Share of Winners by Decile", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([f"D{d}" for d in DECILES])
ax.set_ylim(0, 115)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%g%%"))
ax.legend(frameon=True)
ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.7)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("figures/fig4_share_of_winners.png", bbox_inches="tight")
plt.show()
print("Figure 4 saved.")

In [ ]:
# ── Figure 5: Aggregate Labour Supply Comparison ─────────────────────────────
scenarios = ["Baseline", "Scheme A\n(Lump-Sum)", "Scheme B\n(Targeted)"]
L_values  = [L_base, L_A, L_B]
colors    = [COLORS["baseline"], COLORS["scheme_a"], COLORS["scheme_b"]]

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(scenarios, L_values, color=colors, edgecolor="white",
              linewidth=0.6, width=0.5)

for bar, val in zip(bars, L_values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{val:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylabel("Aggregate Labour Supply (Σ lᵢ*)")
ax.set_title("Figure 5: Aggregate Labour Supply by Scenario", fontweight="bold")
ax.set_ylim(0, max(L_values) * 1.12)
ax.yaxis.grid(True, color=COLORS["grid"], linewidth=0.7)
ax.set_axisbelow(True)

# Annotate differences
diff_A = L_A - L_base
diff_B = L_B - L_base
ax.annotate(f"Δ = {diff_A:+.2f}", xy=(1, L_A), xytext=(1.3, L_A + 1),
            fontsize=9, color=COLORS["scheme_a"],
            arrowprops=dict(arrowstyle="->", color=COLORS["scheme_a"], lw=1.2))
ax.annotate(f"Δ = {diff_B:+.2f}", xy=(2, L_B), xytext=(2.3, L_B + 1),
            fontsize=9, color=COLORS["scheme_b"],
            arrowprops=dict(arrowstyle="->", color=COLORS["scheme_b"], lw=1.2))

plt.tight_layout()
plt.savefig("figures/fig5_aggregate_labour_supply.png", bbox_inches="tight")
plt.show()
print("Figure 5 saved.")

In [ ]:
# ── Figure 6: Equity-Efficiency Frontier (λ sensitivity) ─────────────────────
# For each λ, take the baseline tau = 0.10 row and plot the frontier

baseline_tau_rows = frontier_df[frontier_df["tau"] == 0.10].sort_values("lam")

fig, ax1 = plt.subplots(figsize=(9, 5.5))
ax2 = ax1.twinx()

line1, = ax1.plot(
    baseline_tau_rows["lam"],
    baseline_tau_rows["welfare_bottom40"],
    color=COLORS["scheme_b"], marker="o", linewidth=2,
    label="Welfare gain (bottom 40%)")

line2, = ax2.plot(
    baseline_tau_rows["lam"],
    baseline_tau_rows["labour_loss"],
    color=COLORS["scheme_a"], marker="s", linewidth=2, linestyle="--",
    label="Labour supply change")

ax1.set_xlabel("Targeting multiplier λ (Scheme B)")
ax1.set_ylabel("Mean welfare gain — bottom 40% (ΔU)", color=COLORS["scheme_b"])
ax2.set_ylabel("Aggregate labour supply change (ΔL)", color=COLORS["scheme_a"])
ax1.set_title("Figure 6: Equity-Efficiency Frontier (λ sensitivity, τ = 0.10)",
              fontweight="bold")

ax1.tick_params(axis="y", colors=COLORS["scheme_b"])
ax2.tick_params(axis="y", colors=COLORS["scheme_a"])

# Combined legend
ax1.legend(handles=[line1, line2], loc="center left", frameon=True)
ax1.xaxis.grid(True, color=COLORS["grid"], linewidth=0.7)
ax1.set_axisbelow(True)

plt.tight_layout()
plt.savefig("figures/fig6_equity_efficiency_frontier.png", bbox_inches="tight")
plt.show()
print("Figure 6 saved.")

---
## Summary of Results

The cell below prints a concise summary of all key findings.

In [ ]:
print("=" * 65)
print("  RESULTS SUMMARY")
print("=" * 65)

print(f"\nParameters: tau={params['tau']}, beta={params['beta']},"
      f" alpha={params['alpha']}, lambda={params['lam']}, N={params['N']}")

print(f"\nGini of simulated wage distribution : {compute_gini(wages):.4f}")

print(f"\nEquilibrium transfers:")
print(f"  Scheme A — universal rebate : T  = {T_A:.6f}")
print(f"  Scheme B — T_low (bottom 40%) = {T_B_low:.6f}")
print(f"  Scheme B — T_high (top 60%)   = {T_B_high:.6f}")

print(f"\nNewton's method check: T = {T_newton:.6f}  (diff = {abs(T_A - T_newton):.2e})")

print(f"\nAggregate labour supply:")
print(f"  Baseline : {L_base:.4f}")
print(f"  Scheme A : {L_A:.4f}   ({L_A - L_base:+.4f} vs baseline)")
print(f"  Scheme B : {L_B:.4f}   ({L_B - L_base:+.4f} vs baseline)")

cutoff      = int(0.40 * params["N"])
bottom_mask = np.arange(params["N"]) < cutoff

dU_A_bot = (np.log(c_A) + params["alpha"]*np.log(1-l_A) - U_base)[bottom_mask].mean()
dU_B_bot = (np.log(c_B) + params["alpha"]*np.log(1-l_B) - U_base)[bottom_mask].mean()

print(f"\nMean welfare gain — bottom 40%:")
print(f"  Scheme A : {dU_A_bot:.6f}")
print(f"  Scheme B : {dU_B_bot:.6f}")

print(f"\nScheme B improves bottom-40% welfare by"
      f" {(dU_B_bot - dU_A_bot) / abs(dU_A_bot) * 100:.1f}%"
      f" more than Scheme A.")

print(f"\nLabour supply efficiency cost of Scheme B vs A:"
      f" {L_B - L_A:+.4f}")

print("\n" + "=" * 65)